In [ ]:
# Import Required Libraries

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df = pd.read_csv(
    "IMDB_Dataset.csv",
    encoding_errors='ignore',
    on_bad_lines='skip',
    engine='python'
)
# Convert Sentiment into Numbers
df = df.dropna()
df.columns = ['review', 'sentiment']
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

df.head()

# Split Dataset

X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Convert Text Data into Numerical Data using TF-IDF

tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english'
)


X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

print("Training Data Shape:", X_train_tfidf.shape)
print("Testing Data Shape:", X_test_tfidf.shape)

# Build Deep Neural Network Model

model = Sequential()

model.add(Dense(128, activation='relu',input_shape=(X_train_tfidf.shape[1],)))
model.add(Dense(62, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Show Model Summary
model.summary()

# Train the Model

history = model.fit(
    X_train_tfidf,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate Model

loss, accuracy = model.evaluate(X_test_tfidf, y_test)

print("Test Accuracy:", accuracy)

# Plot Accuracy Graph

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("Accuracy Graph")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


# Test Custom Review

review = ["This movie is very good"]

review_tfidf = tfidf.transform(review).toarray()

result = model.predict(review_tfidf)

if result > 0.5:
    print("Positive Review")
else:
    print("Negative Review")
